# Análisis de la API de Hilos (Pthreads)

## Laboratorio de Sistemas Operativos
### Práctica 4 – API de Hilos

Este notebook contiene los resultados y análisis solicitados en la guía para las aplicaciones **pi_p.c** y **fibonacci.c**.


# Sección 1: Análisis de π

## 1. Evaluación de Ts (Tiempo Serial)

Se ejecutó:

```bash
./pi 2000000000
```

**Ts = 6.718851 s**


## 2. Evaluación de Tp (Tiempo Paralelo)

Resultados experimentales:

| N (Hilos) | Tp (s) |
|-----------|---------|
| 1 | 6.368858 |
| 2 | 2.582942 |
| 4 | 1.951420 |
| 8 | 2.124512 |
| 16 | 2.389145 |


In [ ]:
import pandas as pd

Ts = 6.718851

datos = {
    "N (Hilos)": [1,2,4,8,16],
    "Tp (s)": [6.368858,2.582942,1.951420,2.124512,2.389145]
}

df = pd.DataFrame(datos)
df["Speedup"] = Ts / df["Tp (s)"]
df["Eficiencia"] = df["Speedup"] / df["N (Hilos)"]
df


## 3. Tabla de Resultados

La tabla anterior corresponde al formato solicitado en la guía:

- Speedup = Ts / Tp
- Eficiencia = Speedup / N


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(df["N (Hilos)"], df["Speedup"], marker="o")
plt.xlabel("Número de hilos")
plt.ylabel("Speedup")
plt.title("Speedup vs Número de Hilos")
plt.grid(True)
plt.show()


## 4. Gráfico de Speedup

La gráfica anterior muestra el comportamiento del speedup respecto al número de hilos.


## 5. Análisis de Resultados

### Comparación entre Tp(1) y Ts

Se obtuvo:

- Ts = 6.718851 s
- Tp(1) = 6.368858 s

La diferencia observada es pequeña y puede atribuirse a variaciones normales de ejecución, optimizaciones del compilador, estado del sistema operativo, comportamiento de caché y ruido experimental.

### Speedup máximo alcanzado

El mejor resultado se obtuvo con 4 hilos:

- Tp = 1.951420 s
- Speedup = 3.443×

Si el sistema posee 4 núcleos físicos, este resultado es coherente con la capacidad real de paralelización del hardware.

### Tendencia de la eficiencia

La eficiencia disminuye a medida que aumenta el número de hilos:

- 2 hilos: ~130 %
- 4 hilos: ~86 %
- 8 hilos: ~40 %
- 16 hilos: ~18 %

Este comportamiento se explica por la Ley de Amdahl, el overhead de creación y sincronización de hilos, los cambios de contexto y la competencia por recursos compartidos.


# Sección 2: Análisis de Fibonacci

## 1. Resultados de Ejecución

```bash
./fibonacci 15

Secuencia de Fibonacci (15 elementos):
0 1 1 2 3 5 8 13 21 34 55 89 144 233 377
```


## 2. Análisis del Diseño

### Cálculo sin uso de hilos para N grande

Una implementación iterativa sin hilos mantiene complejidad temporal O(N).

Para valores superiores a 100 000 elementos el problema principal no es el tiempo de ejecución, sino el desbordamiento aritmético de tipos enteros de 64 bits. A partir de términos relativamente pequeños de la serie, los valores exceden la capacidad de almacenamiento del tipo de dato.

### Transferencia de datos al hilo trabajador

Los argumentos se encapsulan en una estructura:

```c
typedef struct {
    long *array;
    int n;
} fib_args_t;
```

El hilo principal crea esta estructura y la pasa como cuarto parámetro de `pthread_create()`. El hilo trabajador recibe un puntero genérico `void *arg`, realiza el cast correspondiente y accede tanto al arreglo compartido como al valor N.

### Rol de pthread_join

`pthread_join()` actúa como mecanismo de sincronización.

El hilo principal queda bloqueado hasta que el hilo trabajador finaliza la generación de la secuencia. Esto garantiza que el arreglo compartido esté completamente lleno antes de ser leído o impreso por el hilo principal.
